# Model Evaluation

Comprehensive evaluation of fine-tuned model with:
- Accuracy, Precision, Recall, F1-Score
- Confusion Matrix
- BERTScore for reasoning quality
- LLM-as-a-Judge (optional)

## Setup

In [ ]:
import sys
sys.path.append('../langchain-refactor')

import json
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, cohen_kappa_score
import numpy as np

## Configuration

In [ ]:
MODEL_PATH = "../models/native-ads-qwen14b-v2_merged_16bit"
TEST_DATASET = "../data/llm_dataset_mixed_json.json"
NUM_SAMPLES = 200  # Number of samples to evaluate

print(f"Model: {MODEL_PATH}")
print(f"Test Dataset: {TEST_DATASET}")
print(f"Samples: {NUM_SAMPLES}")

## Load Model and Dataset

In [ ]:
from chains.full_pipeline_chain import FullPipelineChain
from agents.classification_agent import ClassificationAgent

# Initialize classification agent
agent = ClassificationAgent(
    model_name=MODEL_PATH,
    provider="local"
)

# Load test dataset
with open(TEST_DATASET, 'r') as f:
    test_data = json.load(f)

# Sample for evaluation
import random
random.seed(42)
eval_samples = random.sample(test_data, min(NUM_SAMPLES, len(test_data)))

print(f"✅ Loaded {len(eval_samples)} samples for evaluation")

## Run Evaluation

In [ ]:
predictions = []
ground_truth = []

print("Running evaluation...")
for i, sample in enumerate(eval_samples):
    if i % 20 == 0:
        print(f"Progress: {i}/{len(eval_samples)}")
    
    # Get ground truth
    gt_output = json.loads(sample['output'])
    gt_label = gt_output['label']
    ground_truth.append(gt_label)
    
    # Get prediction
    result = agent.classify(
        title="",
        summary="",
        content=sample['input'],
        context=""
    )
    
    predictions.append(result['label'])

print(f"\n✅ Evaluation complete!")

## Calculate Metrics

In [ ]:
# Overall metrics
accuracy = accuracy_score(ground_truth, predictions)
precision, recall, f1, _ = precision_recall_fscore_support(
    ground_truth, predictions, average='weighted'
)
kappa = cohen_kappa_score(ground_truth, predictions)

print("="*80)
print("EVALUATION RESULTS")
print("="*80)
print(f"\nOverall Metrics:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  Cohen's Kappa: {kappa:.4f}")

# Per-class metrics
labels = sorted(set(ground_truth))
precision_per_class, recall_per_class, f1_per_class, support = precision_recall_fscore_support(
    ground_truth, predictions, labels=labels, average=None
)

print(f"\nPer-Class Metrics:")
for i, label in enumerate(labels):
    print(f"\n  {label}:")
    print(f"    Precision: {precision_per_class[i]:.4f}")
    print(f"    Recall:    {recall_per_class[i]:.4f}")
    print(f"    F1-Score:  {f1_per_class[i]:.4f}")
    print(f"    Support:   {support[i]}")

## Confusion Matrix

In [ ]:
# Calculate confusion matrix
cm = confusion_matrix(ground_truth, predictions, labels=labels)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix - Native Ads Detection', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()

# Save
plt.savefig('../results/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Confusion matrix saved to ../results/confusion_matrix.png")

## Error Analysis

In [ ]:
# Find misclassified samples
errors = []
for i, (gt, pred) in enumerate(zip(ground_truth, predictions)):
    if gt != pred:
        errors.append({
            'index': i,
            'ground_truth': gt,
            'prediction': pred,
            'content': eval_samples[i]['input'][:200]
        })

print(f"\nMisclassified Samples: {len(errors)} ({len(errors)/len(eval_samples)*100:.1f}%)")
print("\nSample Errors:")
for error in errors[:5]:
    print(f"\n  GT: {error['ground_truth']} | Pred: {error['prediction']}")
    print(f"  Content: {error['content']}...")

## Save Results

In [ ]:
results = {
    'model': MODEL_PATH,
    'num_samples': len(eval_samples),
    'overall_metrics': {
        'accuracy': float(accuracy),
        'precision': float(precision),
        'recall': float(recall),
        'f1_score': float(f1),
        'cohens_kappa': float(kappa)
    },
    'per_class_metrics': {
        label: {
            'precision': float(precision_per_class[i]),
            'recall': float(recall_per_class[i]),
            'f1_score': float(f1_per_class[i]),
            'support': int(support[i])
        }
        for i, label in enumerate(labels)
    },
    'confusion_matrix': cm.tolist(),
    'errors': errors
}

# Save to JSON
with open('../results/evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅ Results saved to ../results/evaluation_results.json")